# ShipmentSure – Exploratory Data Analysis

**Dataset:** E-Commerce Shipping Dataset (10,999 rows × 12 columns)  
**Target:** `Reached.on.Time_Y.N` → `on_time_delivery` (1 = on time, 0 = delayed)

---

Sections:
1. Setup & Load
2. Data Overview
3. Missing Values
4. Class Imbalance
5. Univariate – Numerical
6. Univariate – Categorical
7. Bivariate vs. Target
8. Engineered Feature Distributions
9. Correlation Heatmap
10. Key Insights

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Allow imports from project root
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.preprocess import load_data, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, ENGINEERED_FEATURES

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13})

FIG_DIR = os.path.join('..', 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print('Setup complete.')

## 1. Load Data

In [ ]:
df = load_data(os.path.join('..', 'data', 'dataset.csv'))
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe(include='all').T.round(2)

## 2. Missing Values

In [ ]:
miss = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(2)
mv = pd.DataFrame({'Count': miss, '%': miss_pct})
print(mv[mv['Count'] > 0] if mv['Count'].sum() > 0 else 'No missing values ✓')

## 3. Class Imbalance

In [ ]:
vc = df['on_time_delivery'].value_counts().sort_index()
labels = ['Delayed (0)', 'On Time (1)']
colors = ['#e02424', '#0e9f6e']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

bars = axes[0].bar(labels, vc.values, color=colors, edgecolor='white', linewidth=1.5)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{int(bar.get_height()):,}', ha='center', fontweight='bold')
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Count')

axes[1].pie(vc.values, labels=labels, autopct='%1.1f%%', colors=colors,
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution (%)')

plt.suptitle('Target: on_time_delivery', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'class_distribution.png'), bbox_inches='tight')
plt.show()

## 4. Univariate – Numerical Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(NUMERICAL_FEATURES):
    axes[i].hist(df[col].dropna(), bins=30, color='#3A86FF', edgecolor='white', alpha=0.85)
    axes[i].set_title(col)
    axes[i].set_ylabel('Count')
plt.suptitle('Univariate – Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'univariate_numerical.png'), bbox_inches='tight')
plt.show()

## 5. Univariate – Categorical Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()
for i, col in enumerate(CATEGORICAL_FEATURES):
    vc2 = df[col].value_counts()
    axes[i].bar(vc2.index, vc2.values, color='#8B5CF6', edgecolor='white', alpha=0.85)
    axes[i].set_title(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=10)
plt.suptitle('Univariate – Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'univariate_categorical.png'), bbox_inches='tight')
plt.show()

## 6. Bivariate – Numerical vs. Target

In [ ]:
palette = {0: '#e02424', 1: '#0e9f6e'}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(NUMERICAL_FEATURES):
    for lbl, colour in palette.items():
        subset = df[df['on_time_delivery'] == lbl][col].dropna()
        axes[i].hist(subset, bins=25, alpha=0.6, color=colour,
                     label='On Time' if lbl == 1 else 'Delayed', edgecolor='white')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)
plt.suptitle('Bivariate: Numerical vs. On-Time Delivery', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'bivariate_numerical.png'), bbox_inches='tight')
plt.show()

## 7. Bivariate – Categorical vs. Target

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, col in enumerate(CATEGORICAL_FEATURES):
    ct = df.groupby(col)['on_time_delivery'].mean().sort_values(ascending=False)
    bars2 = axes[i].bar(ct.index, ct.values * 100, color='#3A86FF', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{col} – On-Time Rate (%)')
    axes[i].set_ylim(0, 100)
    axes[i].set_ylabel('%')
    axes[i].tick_params(axis='x', rotation=10)
    for bar in bars2:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
plt.suptitle('Bivariate: Categorical vs. On-Time Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'bivariate_categorical.png'), bbox_inches='tight')
plt.show()

## 8. Engineered Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(ENGINEERED_FEATURES):
    for lbl, colour in palette.items():
        subset = df[df['on_time_delivery'] == lbl][col]
        axes[i].hist(subset, bins=20, alpha=0.6, color=colour,
                     label='On Time' if lbl == 1 else 'Delayed', edgecolor='white')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)
plt.suptitle('Engineered Features vs. On-Time Delivery', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'engineered_features.png'), bbox_inches='tight')
plt.show()

## 9. Correlation Heatmap

In [ ]:
corr_cols = NUMERICAL_FEATURES + ENGINEERED_FEATURES + ['on_time_delivery']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', vmin=-1, vmax=1,
            linewidths=.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'correlation_heatmap.png'), bbox_inches='tight')
plt.show()

## 10. Key Insights

- **Class balance:** ~60% delayed, ~40% on time → models trained with `class_weight='balanced'`.
- **Discount_offered:** Large discount (>20%) correlates strongly with delays — `is_high_discount` is a key feature.
- **Weight_in_gms:** Bimodal — light products (~1k–4k g) vs. heavy (~5k–7k g). `is_heavy` captures this split.
- **Customer_care_calls:** More calls strongly correlates with delays — `high_calls` is informative.
- **Mode_of_Shipment:** Flight tends to have a higher on-time rate; Ship is most common.
- **Warehouse_block:** Some blocks show notably different on-time rates.
- **No missing values** in this dataset — imputers are safety measures.